# 01. PubChem 기반 데이터 수집

이 실습에서는 **PubChem REST API (PUG-REST)** 를 이용해 URL 요청만으로
화합물(Chemical) 정보와 생리활성(BioAssay) 데이터를 직접 수집한다.

## PUG-REST 란?
PubChem의 **P**ower **U**ser **G**ateway - **REST** 인터페이스로,
웹 주소(URL)를 조합해 GET 요청을 보내면 JSON/CSV/PNG 형태로 데이터를 돌려준다.
별도의 라이브러리 없이 `requests` 만으로 사용할 수 있다는 것이 장점이다.

```
https://pubchem.ncbi.nlm.nih.gov/rest/pug / <입력> / <조회 대상> / <출력형식>
                     (base)                  (input)   (operation)    (output)
```

- 공식 문서: https://pubchem.ncbi.nlm.nih.gov/docs/pug-rest

## 0. 환경 설정

실습에 필요한 라이브러리를 불러온다. (Google Colab에는 모두 기본 설치되어 있음)
- `requests` : REST API에 HTTP 요청을 보내는 라이브러리
- `pandas`   : 수집한 데이터를 표(DataFrame) 형태로 다루기 위한 라이브러리
- `time`     : PubChem 서버 부하 방지를 위한 요청 간 대기

In [ ]:
import requests           # REST API(URL) 요청
import pandas as pd       # 데이터프레임 처리
import time               # 요청 간 간격 조절
from io import StringIO
from pprint import pprint

---
# 1. PubChem Chemical(Compound) 데이터 수집

> **[실습] REST API 기반 (url) PubChem Chemical 데이터 수집**

PubChem의 화합물 페이지(예: [Aspirin, CID 2244](https://pubchem.ncbi.nlm.nih.gov/compound/2244))
에 정리된 정보(분자식, 분자량, 별명, 구조, 설명 등)를 API로 그대로 가져와 본다.

### Compound 요청 URL 구조
```
{BASE}/compound/{입력방식}/{값}/{조회대상}/{출력형식}
```
| 구성 | 예시 | 설명 |
|------|------|------|
| 입력방식(input) | `name`, `cid`, `smiles` | 무엇으로 찾을지 |
| 값 | `aspirin`, `2244` | 검색 값 |
| 조회대상(operation) | `cids`, `property/...`, `synonyms`, `description` | 무엇을 가져올지 |
| 출력형식(output) | `JSON`, `CSV`, `PNG`, `TXT` | 어떤 형식으로 받을지 |

## 1-1. 화합물 이름 → CID 조회

CID(Compound ID)는 PubChem이 화합물마다 부여하는 고유 번호다.
먼저 화합물 **이름(name)** 으로 CID를 찾아본다. (Aspirin → 2244)

In [ ]:
# PUG-REST 서비스의 기본 주소(base URL). 이후 모든 요청은 이 주소 뒤에 경로를 붙여 만든다.
# 'https://pubchem.ncbi.nlm.nih.gov/rest/pug'

# REST의 규칙에 따라 asiprin에 대해서 'name' 으로 'cid'를 조회하는 url 주소
# name → CID : 기본주소/compound/name/{이름}/cids/JSON
url = 'https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/aspirin/cids/JSON'

# url 주소에 대해서 GET 요청 전송
response = requests.get(url, verify=False)

# 요청에 대한 응답(JSON 파일형태)을 파이썬 dict로 변환 및 받아오기
data = response.json()

pprint(data)

In [ ]:
# cid의 정보는 'IdentifierList' - 'CID' - 첫번째 값
cid = data['IdentifierList']['CID'][0]
print('Asiprin 의 CID =', cid)

## 1-2. 화합물 속성(Property) 수집

슬라이드의 **Molecular Formula / Molecular Weight** 등에 해당하는 값들을 가져온다.
여러 속성을 쉼표로 이어서 한 번에 요청할 수 있다.

- `MolecularFormula` : 분자식 (예: C9H8O4)
- `MolecularWeight`  : 분자량 (예: 180.16 g/mol)
- `IUPACName`        : IUPAC 명명법 이름
- `SMILES`           : 구조를 나타내는 문자열(SMILES)

In [ ]:
# 가져올 속성(Property) 목록
# MolecularFormula / MolecularWeight / IUPACName / SMILES

# REST의 규칙에 따라 asiprin에 대해서 'cid' 으로 'Property'를 조회하는 url 주소
# cid → property : 기본주소/compound/cid/{CID}/property/{속성1,속성2,속성3,...}/JSON
url = 'https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/aspirin/property/MolecularFormula,MolecularWeight,IUPACName,SMILES/JSON'

# url 주소에 대해서 GET 요청 전송
response = requests.get(url, verify=False)

# 요청에 대한 응답(JSON 파일형태)을 파이썬 dict로 변환 및 받아오기
data = response.json()
pprint(data)


In [ ]:
# properties의 정보는 'PropertyTable' - 'Properties' - 첫번째 값
properties = data['PropertyTable']['Properties'][0]
pprint(properties)

In [ ]:
print('Asiprin의 MF:', properties['MolecularFormula'],
      ', MolecualrWeight:', properties['MolecularWeight'],
      ', IUPAC Name:', properties['IUPACName'],
      ', SMILES:', properties['SMILES'])

# properties(dict)를 pandas DataFrame(표)으로 변환
#   - 딕셔너리(dict)를 리스트로 감싸면([ ... ]) 한 줄짜리 표가 만들어진다.
df_property = pd.DataFrame([properties])
df_property

## 1-3. [실습] 이름(name) → 속성(Property) 직접 조회하기

> **직접 해보기 🖐️**

앞의 **1-1**에서는 이름으로 CID를 먼저 찾고, **1-2**에서 속성을 가져왔다.
이번에는 CID를 거치지 않고 **화합물 이름만으로 곧바로 속성**을 조회해 본다.

### 요청 URL 구조
```
{BASE}/compound/name/{이름}/property/{속성1,속성2,...}/JSON
```

**미션** — 아래 코드 셀의 빈칸(`______`)을 채워 Aspirin의 속성을 출력해 보자.
| 가져올 속성 | 의미 |
|------|------|
| `MolecularFormula` | 분자식 |
| `MolecularWeight`  | 분자량 |
| `IUPACName`        | IUPAC 이름 |
| `SMILES`           | 구조 문자열 |

> 💡 막히면 **1-2 셀의 코드**를 그대로 참고하면 된다.

In [ ]:
# =====================================================================
# [실습] 화합물 '이름(name)'으로 곧바로 속성(Property) 조회하기
#   - 위 1-2 셀은 name → property 를 이미 사용했다. 직접 처음부터 작성해 보자.
#   - 아래 밑줄(______) 부분을 채우면 된다. (막히면 위의 코드 참조!)
# =====================================================================

# [1] 요청 URL 만들기
#     구조 : 기본주소/compound/name/{이름}/property/{속성1,속성2,...}/JSON
#     조회할 속성 : MolecularFormula, MolecularWeight, IUPACName, SMILES
url = 'https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/______/property/______/JSON'

# [2] GET 요청 보내기
response = requests.get(url, verify=False)

# [3] 응답(JSON)을 파이썬 dict 로 변환
data = response.json()

# [4] properties 정보만 꺼내기 : 'PropertyTable' → 'Properties' → 첫 번째 값
properties = data['______']['______'][0]

# [5] 결과 출력
pprint(properties)
df_property = pd.DataFrame([properties])
df_property

---
# 2. PubChem BioAssay 데이터 수집

> **[실습] REST API 기반 (url) PubChem BioAssay 데이터 수집**
>
> PubChem REST API BioAssay 로부터 chemical 및 활성 데이터 수집

**BioAssay** 는 화합물의 생리활성 실험 결과를 모아 둔 데이터로,
각 실험은 **AID(Assay ID)** 라는 고유 번호를 가진다.

이번 실습 대상은 슬라이드와 동일하게
[**AID 651580** — *Single concentration confirmation of uHTS identification of HIF-2a Inhibitors*](https://pubchem.ncbi.nlm.nih.gov/bioassay/651580)
이다. (Target: HIF-2a / endothelial PAS domain-containing protein 1)

### Assay 요청 URL 구조
```
{BASE}/assay/aid/{AID}/{조회대상}/{출력형식}
```
- `description` : 어세이 개요(제목, 타겟, 출처 등)
- `cids`        : 실험된 화합물 CID 목록 (전체/Active/Inactive)
- `CSV`         : 화합물별 활성 결과 데이터 표

## 2-1. Assay 개요(Description) 수집

슬라이드의 **Protein Target / Source / BioAssay Type / Description** 등
어세이의 기본 정보를 가져온다.

In [ ]:
# PUG-REST 서비스의 기본 주소(base URL). 이후 모든 요청은 이 주소 뒤에 경로를 붙여 만든다.
# 'https://pubchem.ncbi.nlm.nih.gov/rest/pug'
# 실습 대상 BioAssay ID (HIF-2a 저해제)
# AID = 651580

# REST의 규칙에 따라 특정 BioAssay(aid = 651580)에 대해서 'AID' 으로 'description'를 조회하는 url 주소
# aid → description : 기본주소/assay/aid/{AID}/description/JSON
url = 'https://pubchem.ncbi.nlm.nih.gov/rest/pug/assay/aid/651580/description/JSON'

# url 주소에 대해서 GET 요청 전송
response = requests.get(url, verify=False)

# 요청에 대한 응답(JSON 파일형태)을 파이썬 dict로 변환 및 받아오기
data = response.json()
pprint(data)

In [ ]:
# Assay의 정보는 'PC_AssayContainer' - 첫 번째값 - 'assay' - ['descr]
assay = data['PC_AssayContainer'][0]['assay']['descr']

print('AID   :', assay['aid']['id'])
print('제목  :', assay['name'])
# 어세이 설명은 여러 줄의 리스트로 제공된다
print('설명  :')
for line in assay.get('description', [])[:5]:
    print('  ', line)

## 2-2. 화합물별 활성 데이터 표(CSV) 수집

출력형식을 `CSV` 로 지정하면 화합물별 활성 결과를 표로 받을 수 있다.
핵심 컬럼은 다음과 같다.
- `PUBCHEM_CID`               : 화합물 ID
- `PUBCHEM_ACTIVITY_OUTCOME`  : 활성 판정 (Active / Inactive)
- `PUBCHEM_ACTIVITY_SCORE`    : 활성 점수

슬라이드의 **Tested Substances: All(1,759) / Active(982) / Inactive(777)** 에 해당하는
원본 데이터다.

In [ ]:
# PUG-REST 서비스의 기본 주소(base URL). 이후 모든 요청은 이 주소 뒤에 경로를 붙여 만든다.
# 'https://pubchem.ncbi.nlm.nih.gov/rest/pug'
# 실습 대상 BioAssay ID (HIF-2a 저해제)
# AID = 651580

# REST의 규칙에 따라 특정 BioAssay(aid = 651580)에 대해서 'AID' 으로 'description'를 조회하는 url 주소
# aid → description : 기본주소/assay/aid/{AID}/description/JSON
url = 'https://pubchem.ncbi.nlm.nih.gov/rest/pug/assay/aid/651580/CSV'

# url 주소에 대해서 GET 요청 전송
response = requests.get(url, verify=False)

# 요청에 대한 응답(CSV)을 pandas 표(DataFrame)로 바로 받아오기
data = StringIO(response.text)
data = pd.read_csv(data)
display(data)


# data = pd.read_csv(url)
# data

In [ ]:
df_assay = data.dropna(subset=['PUBCHEM_CID']).reset_index(drop=True)
df_assay.head(5)

In [ ]:
# 활성 판정 분포 확인 (슬라이드의 Active/Inactive 개수와 비교)
df_assay['PUBCHEM_ACTIVITY_OUTCOME'].value_counts()

## 2-3. 활성 데이터에 화합물 구조(SMILES) 결합

활성 데이터(`df_assay`)에는 데이터 제공자가 올린 원본 SMILES(`PUBCHEM_EXT_DATASOURCE_SMILES`)만 있고,
**PubChem이 표준화한 SMILES는 없다.**
원본 SMILES는 표기가 제각각이라, 머신러닝에는 CID로 받은 **표준 SMILES**를 쓰는 것이 좋다.
그래서 각 CID의 표준 SMILES를 받아 활성 데이터와 합친다.

In [ ]:
# [0] 활성 데이터에서 유효한 CID만 추출 (결측 제거 후 정수형 변환)
CIDs = df_assay['PUBCHEM_CID'].dropna().astype(int).unique()
target_CIDs = CIDs[:200].tolist()

print('SMILES를 수집할 CID 개수:', len(target_CIDs))

# [1] CID → SMILES 를 담을 빈 딕셔너리 준비
cid_to_smiles = {}

# [2] for loop (반복문) 을 이용해 추출한 CID의 SMILES 요청 (1-2 참조)
for i in range(0, 10):
    cid = target_CIDs[i]

    url = f'https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/property/SMILES/TXT'

    response = requests.get(url, verify=False)
    cid_to_smiles[cid] = response.text.strip()

    time.sleep(0.2)

print(cid_to_smiles)


In [ ]:
# [3] df_assay에 새 열로 붙이기
df_assay['PUBCHEM_SMILES'] = df_assay['PUBCHEM_CID'].map(cid_to_smiles)
df_assay.head()

## 2-4. 데이터 저장

수집한 데이터셋을 CSV 파일로 저장한다.
이 파일은 다음 강의 **`02_preprocessing_rdkit.ipynb`** 에서 전처리에 사용한다.

In [ ]:
out_path = 'hif2a_aid651580_dataset_demo.csv'
df_assay.to_csv(out_path, index=False)
print('저장 완료:', out_path)

# (Colab) 파일 내려받기 - 필요 시 주석 해제
# from google.colab import files
# files.download(out_path)